# 네이버 뉴스 기사 주소 수집 (Selenium 로컬용)

크롬 창을 직접 띄워서 언론사별 날짜 목록 페이지를 넘기며 기사 주소를 모으는 노트북임.
Colab용 `언론사_네이버뉴스_url_수집_colab.ipynb`의 로컬 버전이고, 파일 이름과 저장 형식은 그대로 맞춰서 다음 단계가 그대로 이어짐.
중간에 끊겨도 중간 저장 파일이 남아 있으면 이어서 수집 가능.

- 입력: `press_ranges`의 언론사명, `oid`, 시작일, 종료일
- 출력: `링크_{press}_{YYMMDD}_{YYMMDD}.json`
- 보조 파일: 수집 기록 JSON, 중간 저장 JSON
- 처리 내용: 날짜별 페이지 확인, 기사 주소 모으기, 중복 기사 제거, 이미 끝난 파일 건너뛰기
- Colab판과 다른 점: requests 대신 크롬 창(Selenium)으로 목록 페이지를 열고, 저장 폴더가 로컬 `data/news/crawling`임


In [ ]:
# # 로컬 커널에 필요한 도구 설치
# # 이미 깔려 있으면 이 셀은 건너뛰어도 됨
# %pip install -q selenium beautifulsoup4 pandas

In [ ]:
import json
import os
import platform
import random
import re
import shutil
import subprocess
import time
from datetime import datetime, timedelta
from pathlib import Path

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

# 네이버에 접속할 때 쓰는 기본 정보 지정 — Colab판, 재시도 노트북과 같은 값 사용
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 크롬 창을 직접 보면서 확인하려면 False, 창 없이 돌리려면 True로 변경
# 로컬에서 눈으로 보려고 만든 노트북이라 기본값은 False
HEADLESS = False

# 수집할 언론사와 기간 지정
# press 하나당 start_date ~ end_date 안의 일자를 한 통합 JSON으로 묶어 저장
# 날짜 형식: 'YYYY.MM.DD'
# oid는 네이버에서 언론사를 구분할 때 쓰는 번호: SBS=055, KBS=056, MBC=214
press_ranges = [
    {'press': 'SBS', 'oid': '055', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': 'KBS', 'oid': '056', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': 'MBC', 'oid': '214', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '한국경제', 'oid': '015', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '매일경제', 'oid': '009', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '한겨레', 'oid': '028', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '조선일보', 'oid': '023', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': '연합뉴스', 'oid': '001', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    {'press': 'YTN',     'oid': '052', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
]


# press_ranges의 항목을 실행할 작업 목록으로 바꿈
# 날짜별 반복은 아래 수집 함수 안에서 처리하므로 여기서는 언론사별 작업만 만듦
def build_period_jobs(press_ranges):
    jobs = []
    for item in press_ranges:
        # 'YYYY.MM.DD' 형식의 날짜를 실제 날짜로 바꿔 기간이 맞는지 확인
        start = datetime.strptime(item['start_date'], '%Y.%m.%d')
        end = datetime.strptime(item['end_date'], '%Y.%m.%d')
        # 시작 일자가 끝 일자보다 늦으면 작업 범위가 잘못된 것이므로 즉시 중단
        if start > end:
            raise ValueError(f"시작 일자가 끝 일자보다 늦습니다: {item}")
        jobs.append({
            'press': item['press'],
            'oid': item['oid'],
            'start_date': item['start_date'],
            'end_date': item['end_date'],
        })
    return jobs


# 만든 작업 목록은 아래 수집 셀에서 순서대로 실행
jobs = build_period_jobs(press_ranges)

print(f'총 작업 수: {len(jobs)}')
for job in jobs:
    print(job)


# 로컬 프로젝트 폴더 찾기 — 노트북을 어느 폴더에서 열어도 프로젝트 최상위를 가리키게 함
# 현재 폴더부터 위로 올라가며 pipeline_py와 notebooks가 같이 있는 곳을 프로젝트 폴더로 봄
DEFAULT_PROJECT_DIR = Path('/home/carol/Text-data-Analysis_26-Spring')


def find_project_dir(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pipeline_py').exists() and (candidate / 'notebooks').exists():
            return candidate
    # 못 찾으면 이 컴퓨터의 기본 경로 사용
    return DEFAULT_PROJECT_DIR


PROJECT_DIR = find_project_dir()

# 저장할 폴더 지정 — 링크 파일, 수집 로그, 중간 저장 파일, 실패 목록이 모두 이 폴더에 저장
# Colab은 Drive의 news/notebook/crawling/data에 저장하는데, 로컬은 data/news/crawling에 따로 모음
# data/news 바로 아래에는 Colab에서 받아 둔 통합본이 있어서 로컬 수집물과 섞이지 않게 한 칸 내려 둠
SAVE_DIR = PROJECT_DIR / 'data' / 'news' / 'crawling'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f'프로젝트 폴더: {PROJECT_DIR}')
print(f'저장 폴더: {SAVE_DIR}')
print(f'User-Agent: {USER_AGENT}')
print(f'HEADLESS: {HEADLESS}')

In [ ]:
# 현재 컴퓨터에 설치된 크롬 위치 찾기
# 크롬 위치를 직접 알려 주면 실행 오류가 줄어듦
def find_chrome_binary():
    candidates = []

    if platform.system() == 'Windows':
        # 윈도우에서 크롬이 자주 설치되는 폴더들을 후보로 둠
        candidates.extend([
            os.path.expandvars(r'%ProgramFiles%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%ProgramFiles(x86)%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%LocalAppData%\Google\Chrome\Application\chrome.exe'),
        ])
    else:
        # 리눅스나 WSL에서는 크롬 이름 후보를 차례로 확인
        for name in ['google-chrome', 'google-chrome-stable', 'chromium-browser', 'chromium']:
            found = shutil.which(name)
            if found:
                candidates.append(found)

    # 실제로 존재하는 첫 번째 경로 사용
    for path in candidates:
        if path and Path(path).exists():
            return str(Path(path))
    return None


# 목록 페이지를 열 크롬 준비
# 기본값은 사람이 화면을 보면서 확인할 수 있도록 창을 띄우는 방식임
def build_driver(headless=HEADLESS):
    options = Options()
    options.add_argument(f'user-agent={USER_AGENT}')  # 수집할 때 쓰는 기본 정보 맞춤
    options.add_argument('--lang=ko-KR')  # 가능하면 한국어 페이지로 받기
    options.add_experimental_option('excludeSwitches', ['enable-automation'])  # 크롬이 자동 실행 창처럼 보이는 표시를 줄임
    options.add_experimental_option('useAutomationExtension', False)  # 크롬이 자동 실행 창처럼 보이는 표시를 줄임
    options.add_argument('--disable-blink-features=AutomationControlled')  # 크롬이 자동 실행 창처럼 보이는 표시를 줄임
    options.add_argument('--window-size=1400,1000')  # 항상 비슷한 화면 크기로 열기

    if headless:
        options.add_argument('--headless=new')  # 창 없이 실행

    if platform.system() != 'Windows':
        options.add_argument('--no-sandbox')  # 리눅스/WSL에서 크롬 실행 오류 줄이기
        options.add_argument('--disable-dev-shm-usage')  # 크롬이 중간에 꺼지는 문제 줄이기
        options.add_argument('--disable-gpu')  # 창 없이 실행할 때 그래픽 관련 오류 줄이기

    # 크롬 실행 파일을 찾으면 직접 지정, 못 찾으면 기본 방식으로 실행
    chrome_binary = find_chrome_binary()
    if chrome_binary:
        options.binary_location = chrome_binary
        print(f'Chrome binary: {chrome_binary}')
        subprocess.run([chrome_binary, '--version'], check=False)
    else:
        print('크롬 실행 파일을 직접 찾지 못해 Selenium 기본 방식으로 실행')

    # 현재 크롬에 맞는 실행 도구로 크롬 창 열기
    driver = webdriver.Chrome(service=Service(), options=options)

    # 네이버가 자동 실행 창이라고 판단할 가능성을 조금 낮춤
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
    })
    return driver


driver = build_driver()

In [ ]:
# 서버에 너무 자주 요청하지 않도록 페이지/날짜/언론사 사이에 랜덤 대기
# PAGE_PAUSE: 같은 날짜 안에서 페이지를 넘길 때 쉬는 시간 (짧게)
# DAY_PAUSE: 같은 기간 안에서 날짜를 바꿀 때 쉬는 시간 (중간)
# JOB_PAUSE: 다른 언론사로 넘어갈 때 쉬는 시간 (길게)
# 크롬 창을 띄우는 방식은 requests보다 페이지 한 장에 시간이 더 걸려서 페이지 대기는 조금 여유 있게 잡음
PAGE_PAUSE_RANGE_SEC = (0.8, 1.8)
DAY_PAUSE_RANGE_SEC = (2, 5)
JOB_PAUSE_RANGE_SEC = (5, 12)
PAGE_LOAD_WAIT_SEC = 0.6  # 페이지가 다 뜬 뒤 화면을 눈으로 볼 짬도 되는 짧은 대기
SELENIUM_WAIT_SEC = 15  # 페이지가 다 열릴 때까지 최대로 기다리는 시간
SKIP_COMPLETED = True  # 최종 링크 파일이 이미 있는 기간은 건너뜀
USE_YEONHAP_EXTENDED_URL = False  # 연합뉴스는 주소 형식이 다를 수 있어 필요할 때만 True로 변경


# 랜덤 대기 후 로그 출력
def polite_sleep(label, pause_range):
    pause_sec = random.uniform(*pause_range)
    print(f"{label} {pause_sec:.1f}초 대기")
    time.sleep(pause_sec)


def build_list_url(oid, date_ymd, page=1):
    # 네이버 언론사별 날짜 목록 주소 생성. date_ymd는 'YYYYMMDD' 모양
    # mode=LPOD는 언론사별 기사 목록을 뜻함
    # 연합뉴스 목록이 기본 주소로 잘 안 열리면 위 옵션을 True로 바꿔 사용
    if USE_YEONHAP_EXTENDED_URL and oid == '001':
        return (
            f'https://news.naver.com/main/list.naver?'
            f'mode=LPOD&sid2=140&sid1=001&mid=sec'
            f'&oid={oid}&isYeonhapFlash=Y&date={date_ymd}&page={page}'
        )
    return (
        f'https://news.naver.com/main/list.naver?mode=LPOD&mid=sec'
        f'&oid={oid}&date={date_ymd}&page={page}'
    )


def get_list_html(driver, oid, date_ymd, page=1):
    # 크롬 창으로 목록 페이지 열기
    # 크롬이 글자 인코딩을 알아서 처리해 주기 때문에 Colab판처럼 euc-kr을 따로 지정할 필요 없음
    driver.get(build_list_url(oid, date_ymd, page))

    # 페이지가 다 열릴 때까지 기다림 — 기사가 하나도 없는 날짜도 있어서 목록 영역이 아니라 문서 상태로 판단
    WebDriverWait(driver, SELENIUM_WAIT_SEC).until(
        lambda d: d.execute_script('return document.readyState') == 'complete'
    )
    time.sleep(PAGE_LOAD_WAIT_SEC)

    # 화면에 뜬 페이지 내용을 그대로 가져와 BeautifulSoup으로 넘김
    return driver.page_source


def detect_max_page(html, oid, date_ymd):
    # 아래쪽 페이지 번호들 중 가장 큰 번호를 마지막 페이지로 사용
    # 페이지 번호를 먼저 확인한 뒤 1페이지부터 마지막 페이지까지 반복
    pattern = re.compile(rf'oid={oid}[^"\s]*?date={date_ymd}[^"\s]*?page=(\d+)')  # 해당 언론사·날짜의 페이지 번호 찾기
    pages = [int(m) for m in pattern.findall(html)]
    return max(pages) if pages else 1  # 페이지 번호를 못 찾으면 1페이지만 있다고 보고 진행


def extract_article_links(html, oid):
    # 네이버 목록에는 강조 기사 영역과 일반 기사 영역이 따로 있으므로 둘 다 확인
    # 두 영역의 기사 링크 중 현재 언론사 기사 주소만 골라냄
    soup = BeautifulSoup(html, 'html.parser')
    links = set()
    # 같은 언론사 기사만 모으고 외부 링크나 공유 링크는 제외
    # 크롬이 넘겨주는 주소는 전체 주소일 수도, '/mnews/...'처럼 앞이 잘린 주소일 수도 있어 둘 다 받음
    article_path_pattern = re.compile(rf'^(?:https://n\.news\.naver\.com)?/mnews/article/{oid}/(\d+)')
    for a in soup.select('ul.type06_headline dt a, ul.type06 dt a'):
        href = a.get('href', '')
        m = article_path_pattern.match(href)
        if m:
            # 주소 뒤의 부가 정보는 떼고 기본 기사 주소만 저장해 중복을 줄임
            # Colab판과 같은 모양으로 다시 만들어서 두 환경 결과가 그대로 섞일 수 있게 함
            links.add(f'https://n.news.naver.com/mnews/article/{oid}/{m.group(1)}')
    return links


# 하루치 모든 페이지에서 기사 링크를 모으고 페이지별 개수도 기록
def collect_links_for_day(driver, oid, date_ymd):
    # 1페이지를 먼저 받아 마지막 페이지 번호를 확인하고, 아래 반복문에서도 그대로 사용
    first_html = get_list_html(driver, oid, date_ymd, page=1)
    max_page = detect_max_page(first_html, oid, date_ymd)

    day_links = set()
    page_stats = []

    # 1 ~ max_page 순회하며 기사 링크 누적
    for page in range(1, max_page + 1):
        if page == 1:
            html = first_html  # 위에서 받아둔 1페이지 내용 다시 사용
        else:
            # 2페이지부터는 페이지 전환 대기 후 새로 요청
            polite_sleep(f"  page {page} 열기 전", PAGE_PAUSE_RANGE_SEC)
            html = get_list_html(driver, oid, date_ymd, page=page)

        page_links = extract_article_links(html, oid)
        before = len(day_links)
        day_links.update(page_links)
        # 페이지별 수집량 기록 — 특정 페이지에서 0건이 나오면 기사 링크 위치가 바뀌었는지 확인
        page_stats.append({
            'page': page,
            'found': len(page_links),
            'added': len(day_links) - before,
            'total_this_day': len(day_links),
        })

    return day_links, max_page, page_stats


# 한 언론사의 기간 전체를 통합 JSON 1개로 저장
# 날짜별 중간 저장 파일로 중단 후 이어서 수집 가능
def collect_links_for_period(press, oid, start_date, end_date, driver=None, save_dir=SAVE_DIR):
    # 크롬 창을 따로 넘기지 않으면 위 셀에서 만들어 둔 driver 사용
    # 창이 꺼져서 build_driver()로 다시 만든 경우에도 새 창을 잡음
    if driver is None:
        driver = globals()['driver']

    # 파일명에 들어가는 기간 접미사 (예: 2026.05.01~2026.05.07 -> 260501_260507)
    start_yymmdd = start_date.replace('.', '')[2:]
    end_yymmdd = end_date.replace('.', '')[2:]
    period = f"{start_yymmdd}_{end_yymmdd}"

    # 파일 경로 — temp는 중간 저장, links는 최종 링크, stats는 수집 기록
    temp_links_path = save_dir / f"{press}_{start_date}_{end_date}_temp_links.json"
    links_save_path = save_dir / f"링크_{press}_{period}.json"
    stats_save_path = save_dir / f"수집로그_{press}_{period}.json"

    # 최종 파일이 이미 있으면 같은 기간은 다시 수집하지 않음
    if SKIP_COMPLETED and links_save_path.exists():
        print()
        print(f"=== {press} / {start_date} ~ {end_date} 이미 완료됨, 건너뜀 ===")
        print(f"기존 파일: {links_save_path}")
        return links_save_path

    print()
    print(f"=== {press} / {start_date} ~ {end_date} 수집 시작 ===")

    # 중간 파일에 기존 링크가 있으면 불러오기 — last_date 다음 날부터 이어서 수집
    if temp_links_path.exists():
        with temp_links_path.open('r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        # 이미 모은 링크와 새 링크가 겹치지 않도록 중복 확인하기 쉬운 형태로 바꿈
        all_links_set = set(checkpoint.get('links', []))
        last_collected_date = checkpoint.get('last_date')
        # daily_stats도 중간 파일에 같이 보관해서 중단 전 로그 유지
        daily_stats = checkpoint.get('daily_stats', [])
        print(f"기존 중간 파일에서 링크 {len(all_links_set)}개 불러옴 — {last_collected_date} 다음부터 이어서 수집")
    else:
        all_links_set = set()
        last_collected_date = None
        daily_stats = []
        print('새로 링크 수집 시작')

    # 시작 ~ 끝 일자를 하루씩 순회
    current = datetime.strptime(start_date, '%Y.%m.%d')
    end = datetime.strptime(end_date, '%Y.%m.%d')

    while current <= end:
        day_str = current.strftime('%Y.%m.%d')

        # 이미 수집한 날짜면 건너뜀 — 중간에 다시 실행해도 같은 날짜를 반복 수집하지 않음
        if last_collected_date and day_str <= last_collected_date:
            print(f"{day_str} — 이미 수집 완료, 건너뜀")
            current += timedelta(days=1)
            continue

        date_ymd = day_str.replace('.', '')  # 네이버 주소에 넣을 날짜 형식
        started_at = time.time()

        # 하루치 모든 페이지 순회해서 링크 추출
        day_links, max_page, page_stats = collect_links_for_day(driver, oid, date_ymd)
        before = len(all_links_set)
        all_links_set.update(day_links)
        added = len(all_links_set) - before  # 일자 간 중복 제거 후 늘어난 개수
        elapsed = round(time.time() - started_at, 2)

        # 날짜별 수집량, 페이지 수, 걸린 시간 기록
        daily_stats.append({
            'date': day_str,
            'found': len(day_links),
            'added': added,
            'total': len(all_links_set),
            'max_page': max_page,
            'elapsed_sec': elapsed,
            'pages': page_stats,
        })

        print(
            f"{day_str} — {len(day_links)}건 수집 / 신규 {added}건 추가 "
            f"/ 누적 {len(all_links_set)}건 / 페이지 {max_page} / {elapsed}초"
        )

        # 하루치 수집 후 중간 파일에 즉시 저장 (중간에 끊겨도 누적 저장 + 마지막 완료 날짜 기록)
        with temp_links_path.open('w', encoding='utf-8') as f:
            json.dump(
                {'links': sorted(all_links_set), 'last_date': day_str, 'daily_stats': daily_stats},
                f, ensure_ascii=False, indent=2,
            )
        last_collected_date = day_str
        current += timedelta(days=1)
        # 다음 날짜로 넘어가기 전 대기 (마지막 날 뒤엔 안 함)
        if current <= end:
            polite_sleep('다음 날짜 전', DAY_PAUSE_RANGE_SEC)

    # 기간 전체 링크를 하나로 합쳐 저장 — 순서를 정리해 저장할 때마다 파일 순서가 흔들리지 않게 함
    naver_news_links = sorted(all_links_set)
    with open(links_save_path, 'w', encoding='utf-8') as f:
        json.dump(naver_news_links, f, ensure_ascii=False, indent=2)

    # 수집 로그 — 일자별 통계 + 전체 요약
    with open(stats_save_path, 'w', encoding='utf-8') as f:
        json.dump({
            'press': press,
            'oid': oid,
            'start_date': start_date,
            'end_date': end_date,
            'total': len(naver_news_links),
            'days': daily_stats,
        }, f, ensure_ascii=False, indent=2)

    # 정상 완료 시 중간 파일 삭제 — 다음 실행 시 예전 중간 파일 오독 방지
    if temp_links_path.exists():
        temp_links_path.unlink()

    print(f"수집 완료 — 총 {len(naver_news_links)}개")
    print(f"링크 저장: {links_save_path}")
    print(f"수집 로그 저장: {stats_save_path}")
    return links_save_path


# 작업 단위로 실행 (작업 하나 = 언론사 × 기간)
# 한 작업이 실패해도 실패 목록에 기록하고 다음 작업으로 넘어감
results = []
failures = []
for index, job in enumerate(jobs, start=1):
    print()
    print(f"[{index}/{len(jobs)}] 작업 실행: {job}")
    try:
        # 작업 정보의 언론사, 번호, 시작일, 종료일을 수집 함수에 전달
        results.append(collect_links_for_period(driver=driver, **job))
    except Exception as exc:
        # 한 언론사에서 오류가 나도 전체 작업이 멈추지 않도록 실패 정보만 저장
        failures.append({'job': job, 'error': repr(exc)})
        print(f"작업 실패, 다음 작업으로 넘어갑니다: {exc!r}")
    finally:
        if index < len(jobs):
            # 다음 언론사로 넘어가기 전 대기
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

# 실패한 작업이 있으면 다시 돌릴 수 있게 파일로 저장
if failures:
    failures_path = SAVE_DIR / '수집실패목록_naver.json'
    with open(failures_path, 'w', encoding='utf-8') as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)
    print()
    print(f"실패 작업 {len(failures)}개 저장: {failures_path}")

print()
print('전체 작업 완료')
print(f'성공/건너뜀: {len(results)}개, 실패: {len(failures)}개')
for result_path in results:
    print(result_path)

In [ ]:
# 작업이 끝나면 크롬 창 종료
# 필요하면 이 셀만 따로 실행
try:
    driver.quit()
    print('브라우저 종료 완료')
except Exception as exc:
    print(f'브라우저 종료 중 오류: {exc!r}')